In [1]:
import importlib
import NeuralNetwork
import funcs
import plots

importlib.reload(NeuralNetwork)
importlib.reload(funcs)
importlib.reload(plots)
from NeuralNetwork import NeuralNetwork

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from collections import defaultdict
import numpy as np
from tqdm.auto import tqdm
import copy
import os

In [ ]:
import importlib
import setup
importlib.reload(setup)
from setup import (
    HIDDEN_LAYERS, MAX_ALLOWED_ACC_DROP, MAX_PRUNE_ROUNDS,
    PRUNE_FRAC, PRUNE_CON_FRAC, REGROW_FRAC, MIN_VAL_ACC, N_RETRIAN_EPOCHS
)

device = setup.get_device()
N_TRAIN_EPOCHS = 15 if device.type == "cuda" else 8
train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = setup.get_dataloaders()

In [ ]:
# Create model and train
model = NeuralNetwork(hidden_sizes=HIDDEN_LAYERS, device=device)
model.train_model(train_loader=train_loader, val_loader=val_loader, epochs=N_TRAIN_EPOCHS)

## Prune Neurons and Retrain

In [ ]:
prune_parameters = (MAX_PRUNE_ROUNDS, PRUNE_FRAC, PRUNE_CON_FRAC, REGROW_FRAC, N_RETRIAN_EPOCHS, MAX_ALLOWED_ACC_DROP)
use_max_rounds = False if device.type == "cuda" else True

final_model = funcs.pruning(model, train_loader, val_loader, prune_parameters, use_max_rounds=use_max_rounds, mode='full')

In [ ]:
print(f"Test accuracy after pruning: {final_model.accuracy(val_loader):.2f}")

In [ ]:
torch.save(final_model, "pruned_model.pth")